# Learning Theory — Finite Hypothesis Class

## Learning Objectives
- Prove **uniform convergence**: $\hat{\varepsilon}(h)$ is close to $\varepsilon(h)$ simultaneously for all $h \in \mathcal{H}$
- Derive the **generalisation bound** for ERM on a finite hypothesis class of size $k$
- Understand **sample complexity**: how many training examples are needed to guarantee a performance level
- See how the bias-variance tradeoff is formalised by the theorem: larger $\mathcal{H}$ → lower best-possible error but larger variance term

## Problem Statement

Consider a **finite** hypothesis class $\mathcal{H} = \{h_1, \ldots, h_k\}$.

We want to bound the generalisation error of the ERM hypothesis $\hat{h} = \arg\min_h \hat{\varepsilon}(h)$.

**Key result (Notes4 §3 Theorem):**  
Let $|\mathcal{H}| = k$. For any fixed $m, \delta$, with probability at least $1 - \delta$:

$$\varepsilon(\hat{h}) \leq \left(\min_{h \in \mathcal{H}} \varepsilon(h)\right) + 2\sqrt{\frac{1}{2m}\log\frac{2k}{\delta}}$$

**Sample complexity corollary:**  
To guarantee $\varepsilon(\hat{h}) \leq \varepsilon(h^*) + 2\gamma$ with probability $1 - \delta$, it suffices to have:

$$m \geq \frac{1}{2\gamma^2} \log \frac{2k}{\delta}$$

Sample complexity is only **logarithmic** in $k$ — a key result.

## 1. Uniform Convergence

### Step-by-step derivation

**Fix $h_i \in \mathcal{H}$.** Define Bernoulli r.v. $Z_j = \mathbf{1}\{h_i(x^{(j)}) \neq y^{(j)}\}$.  
Then $\varepsilon(h_i) = \mathbb{E}[Z_j]$ and $\hat{\varepsilon}(h_i) = \frac{1}{m}\sum_j Z_j$.

By Hoeffding:
$$P(|\varepsilon(h_i) - \hat{\varepsilon}(h_i)| > \gamma) \leq 2\exp(-2\gamma^2 m)$$

**For all $h \in \mathcal{H}$ simultaneously** — apply the Union Bound over $k$ events:
$$P\bigl(\exists h \in \mathcal{H}: |\varepsilon(h) - \hat{\varepsilon}(h)| > \gamma\bigr) \leq 2k\exp(-2\gamma^2 m)$$

Complementing: with probability at least $1 - 2k\exp(-2\gamma^2 m)$:
$$\forall h \in \mathcal{H}: |\varepsilon(h) - \hat{\varepsilon}(h)| \leq \gamma \qquad \text{(uniform convergence)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Simulate k hypotheses, each with a true error rate drawn from [0.1, 0.5]
# For each, sample m training examples and compare train error to true error
k = 10
true_epsilons = np.random.uniform(0.1, 0.5, k)
m_values = [10, 50, 200, 1000]
n_trials = 3000
gamma    = 0.1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: show gaps |ε - ε̂| for each h_i at different m
ax = axes[0]
for m in m_values:
    gaps = []
    for eps in true_epsilons:
        hat_eps = np.random.binomial(m, eps, n_trials) / m
        gaps.append(np.abs(hat_eps - eps).mean())
    ax.plot(range(1, k+1), gaps, '-o', ms=5, label=f'$m={m}$')

ax.axhline(gamma, color='r', ls='--', lw=1.5, label=f'$\\gamma={gamma}$')
ax.set_xlabel('Hypothesis index $i$'); ax.set_ylabel('Mean $|\\varepsilon(h_i) - \\hat{\\varepsilon}(h_i)|$')
ax.set_title(f'Average gap $|\\varepsilon - \\hat{{\\varepsilon}}|$ per hypothesis\n($k={k}$ hypotheses, varied $m$)')
ax.legend(fontsize=9)

# Right: P(any h violates) vs m, compared to Hoeffding union bound
ax = axes[1]
m_range = np.logspace(1, 4, 80).astype(int)
empirical_any = []
union_bound   = []

for m in m_range:
    violations_per_trial = []
    for _ in range(500):
        hat_eps = np.array([np.random.binomial(m, e) / m for e in true_epsilons])
        violations_per_trial.append(np.any(np.abs(hat_eps - true_epsilons) > gamma))
    empirical_any.append(np.mean(violations_per_trial))
    union_bound.append(min(2 * k * np.exp(-2 * gamma**2 * m), 1.0))

ax.semilogy(m_range, empirical_any, 'b-',  lw=2.5,
            label=r'Simulated $P(\exists h: |\varepsilon-\hat{\varepsilon}|>\gamma)$')
ax.semilogy(m_range, union_bound,   'r--', lw=2.5,
            label=r'Union bound $2ke^{-2\gamma^2 m}$')
ax.set_xlabel('Sample size $m$'); ax.set_ylabel('Probability (log scale)')
ax.set_title(f'Uniform Convergence: $k={k}$, $\\gamma={gamma}$')
ax.legend(fontsize=9)

fig.suptitle('Uniform Convergence — Empirical Verification', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Generalisation Bound for ERM

Assuming uniform convergence holds (i.e. $|\varepsilon(h) - \hat{\varepsilon}(h)| \leq \gamma$ for all $h$), let $h^* = \arg\min_{h \in \mathcal{H}} \varepsilon(h)$ be the best hypothesis. Then:

$$\varepsilon(\hat{h}) \leq \hat{\varepsilon}(\hat{h}) + \gamma \leq \hat{\varepsilon}(h^*) + \gamma \leq \varepsilon(h^*) + 2\gamma$$

The three inequalities use:
1. Uniform convergence on $\hat{h}$
2. ERM property: $\hat{h}$ minimises $\hat{\varepsilon}$, so $\hat{\varepsilon}(\hat{h}) \leq \hat{\varepsilon}(h^*)$
3. Uniform convergence on $h^*$

Setting $\delta = 2k\exp(-2\gamma^2 m)$ and solving for $\gamma$ gives the theorem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

# Verify the bound numerically: for a finite H, does ε(ĥ) ≤ ε(h*) + 2γ hold?
k          = 20
n_trials   = 2000
gamma      = 0.1
true_eps   = np.random.uniform(0.05, 0.45, k)
eps_star   = true_eps.min()
bound_add  = 2 * gamma

m_values = [20, 50, 100, 300, 500, 1000]
mean_eps_hhat  = []
exceed_bound   = []     # fraction of trials where ε(ĥ) > ε(h*) + 2γ

for m in m_values:
    eps_hhats = []
    exceedances = 0
    for _ in range(n_trials):
        # Simulate ERM: pick h with smallest empirical error
        hat_eps = np.array([np.random.binomial(m, e) / m for e in true_eps])
        best_idx = np.argmin(hat_eps)
        eps_hhat = true_eps[best_idx]
        eps_hhats.append(eps_hhat)
        if eps_hhat > eps_star + bound_add:
            exceedances += 1
    mean_eps_hhat.append(np.mean(eps_hhats))
    exceed_bound.append(exceedances / n_trials)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: ε(ĥ) vs m
ax = axes[0]
ax.plot(m_values, mean_eps_hhat, 'b-o', lw=2.5, ms=7, label=r'Mean $\varepsilon(\hat{h})$')
ax.axhline(eps_star,                color='g',  ls='-',  lw=2,   label=r'$\varepsilon(h^*) = $' + f'{eps_star:.3f}')
ax.axhline(eps_star + bound_add,    color='r',  ls='--', lw=1.5, label=r'$\varepsilon(h^*)+2\gamma$')
ax.set_xlabel('Sample size $m$'); ax.set_ylabel(r'Generalisation error $\varepsilon(\hat{h})$')
ax.set_title(f'ERM converges to $h^*$ as $m$ grows\n($k={k}$, $\\gamma={gamma}$)')
ax.legend(fontsize=9)

# Right: fraction of trials where bound is violated
ax = axes[1]
ax.semilogy(m_values, [max(e, 1e-5) for e in exceed_bound], 'r-s', lw=2.5, ms=7,
            label=r'Fraction trials: $\varepsilon(\hat{h}) > \varepsilon(h^*)+2\gamma$')
theoretical_delta = [min(2 * k * np.exp(-2 * gamma**2 * m), 1.0) for m in m_values]
ax.semilogy(m_values, theoretical_delta, 'b--', lw=2,
            label=r'Theoretical $\delta = 2ke^{-2\gamma^2 m}$')
ax.set_xlabel('Sample size $m$'); ax.set_ylabel('Probability (log scale)')
ax.set_title('Bound violation rate vs. theoretical $\\delta$')
ax.legend(fontsize=9)

fig.suptitle('Generalisation Bound Verification for ERM on Finite $\\mathcal{H}$',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Sample Complexity

**How many training examples do we need?**

Setting $\delta = 2k\exp(-2\gamma^2 m)$ and solving for $m$:

$$m \geq \frac{1}{2\gamma^2} \log \frac{2k}{\delta} = O\!\left(\frac{1}{\gamma^2} \log \frac{k}{\delta}\right)$$

Key observations:
- **Logarithmic in $k$**: even if $\mathcal{H}$ is huge, we don't need exponentially more data — just $\log k$ more.
- **Inverse quadratic in $\gamma$**: halving the desired accuracy gap quadruples the required data.
- **Logarithmic in $1/\delta$**: going from 95% to 99% confidence requires only a small constant factor more data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

delta = 0.05

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: m vs k (for fixed γ, δ)
ax = axes[0]
k_range = np.logspace(0, 6, 100)
for gamma_val, ls in [(0.05, '-'), (0.1, '--'), (0.2, ':')]:
    m_req = (1 / (2 * gamma_val**2)) * np.log(2 * k_range / delta)
    ax.loglog(k_range, m_req, ls, lw=2, label=f'$\\gamma={gamma_val}$')
ax.set_xlabel('Hypothesis class size $k$')
ax.set_ylabel('Required sample size $m$')
ax.set_title(f'Sample complexity vs. $k$ ($\\delta={delta}$)\nLogarithmic growth in $k$')
ax.legend(fontsize=9)

# Middle: m vs γ (for fixed k, δ)
ax = axes[1]
gamma_range = np.linspace(0.02, 0.5, 200)
for k_val, ls in [(10, '-'), (100, '--'), (10000, ':')]:
    m_req = (1 / (2 * gamma_range**2)) * np.log(2 * k_val / delta)
    ax.plot(gamma_range, m_req, ls, lw=2, label=f'$k={k_val}$')
ax.set_xlabel('Accuracy parameter $\\gamma$')
ax.set_ylabel('Required sample size $m$')
ax.set_title(f'Sample complexity vs. $\\gamma$ ($\\delta={delta}$)\nInverse quadratic in $\\gamma$')
ax.legend(fontsize=9)
ax.set_ylim(0, 3000)

# Right: m vs δ (for fixed k, γ)
ax = axes[2]
delta_range = np.linspace(0.001, 0.5, 200)
gamma_val   = 0.1
for k_val, ls in [(10, '-'), (100, '--'), (10000, ':')]:
    m_req = (1 / (2 * gamma_val**2)) * np.log(2 * k_val / delta_range)
    ax.plot(delta_range, m_req, ls, lw=2, label=f'$k={k_val}$')
ax.set_xlabel('Confidence parameter $\\delta$ (failure probability)')
ax.set_ylabel('Required sample size $m$')
ax.set_title(f'Sample complexity vs. $\\delta$ ($\\gamma={gamma_val}$)\nLogarithmic in $1/\\delta$')
ax.legend(fontsize=9)

fig.suptitle('Sample Complexity $m \\geq \\frac{1}{2\\gamma^2}\\log\\frac{2k}{\\delta}$',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="464"
     viewBox="0 0 780 464" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Finite hypothesis class -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Finite hypothesis</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">class |&#x1d4d7;| = k</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >each h_i corresponds to one Bernoulli error r.v. per training example</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >Hoeffding per h_i: P(|&#x3b5;&#x2212;&#x3b5;&#x302;|&gt;&#x3b3;) &#x2264; 2e&#x207b;&#xb2;&#x3b3;&#xb2;m</text>

  <!-- Row 2: Hoeffding per hypothesis -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Hoeffding per h_i</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >each individual hypothesis is well-estimated by its training error</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >union bound: P(&#x2203;h violates) &#x2264; 2k&#xb7;e&#x207b;&#xb2;&#x3b3;&#xb2;m</text>

  <!-- Row 3: Uniform convergence -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Uniform</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">convergence</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >|&#x3b5;(h)&#x2212;&#x3b5;&#x302;(h)| &#x2264; &#x3b3; simultaneously for ALL h &#x2208; &#x1d4d7; w.p. &#x2265; 1&#x2212;&#x3b4;</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >chain: &#x3b5;(&#x125;)&#x2264;&#x3b5;&#x302;(&#x125;)+&#x3b3;&#x2264;&#x3b5;&#x302;(h*)+&#x3b3;&#x2264;&#x3b5;(h*)+2&#x3b3;</text>

  <!-- Row 4: Generalisation bound -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Generalisation</text>
  <text x="102" y="352" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">bound</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >&#x3b5;(&#x125;) &#x2264; &#x3b5;(h*) + 2&#x221a;(log(2k/&#x3b4;) / 2m) with prob. &#x2265; 1&#x2212;&#x3b4;</text>

  <!-- step 4&#x2192;5 -->
  <line x1="102" y1="358" x2="102" y2="408"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 5: Sample complexity -->
  <rect x="10" y="412" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="440" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Sample complexity</text>
  <line x1="197" y1="435" x2="216" y2="435"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="412" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >m &#x2265; (1/2&#x3b3;&#xb2;) log(2k/&#x3b4;) &#x2014; logarithmic in k, inverse quadratic in &#x3b3;</text>
</svg>
""")

## Summary

| Concept | Formula / Statement | Key Insight |
|---|---|---|
| Uniform convergence | $\forall h \in \mathcal{H}: \lvert\varepsilon(h) - \hat{\varepsilon}(h)\rvert \leq \gamma$ | Holds simultaneously for all $h$ with high probability |
| Probability of violation | $P(\exists h: \lvert\varepsilon - \hat{\varepsilon}\rvert > \gamma) \leq 2k e^{-2\gamma^2 m}$ | Union bound + Hoeffding |
| Generalisation bound (finite $\mathcal{H}$) | $\varepsilon(\hat{h}) \leq \varepsilon(h^*) + 2\sqrt{\frac{\log(2k/\delta)}{2m}}$ | Holds with probability $\geq 1 - \delta$ |
| Sample complexity | $m \geq \frac{1}{2\gamma^2}\log\frac{2k}{\delta}$ | Logarithmic in $k$; inverse quadratic in $\gamma$ |
| Bias term | $\min_{h \in \mathcal{H}} \varepsilon(h) = \varepsilon(h^*)$ | Error of best possible $h$ in the class |
| Variance term | $2\sqrt{\frac{\log(2k/\delta)}{2m}}$ | Decreases with $m$; increases with $\log k$ |

**Key insight:** the generalisation error of ERM is bounded by the best-in-class error plus a variance term that shrinks as $O(1/\sqrt{m})$ — and grows only logarithmically with the size of the hypothesis class.